## Linear Regression

There are two main functions to perform linear regression in `scikit-learn`.
- `SGDRegressor` using gradient descent. This uses `gradient descent approach` to optimize a loss function. This approach allows you to incorporate regularization techniques.
- `LinearRegression` using least square method. This uses simple matrix algebra. The function doesn't allow you to incorporate regularization.

You can also use `statsmodels` to perfrom least square method for linear regression. It gives a good summary of the model fit.

You can also use `TensorFlow` to perform linear regression. `Tensorflow` uses `gradient descent` approach and allows regularization. But you also have the option of using different optimization algorithms.

Note: you have to be careful when using `SGDRegressor` and `TensorFlow`. There are many parameters the functions take and understanding the parameters is important. More on this later.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import scale
import sklearn.linear_model as skl_lm
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
import statsmodels.formula.api as smf


### Load Datasets

This is the Advertising data set.
- Sales column is the response variable. Represents number of units sold (per 1000)
- TV, Radio, and Newpaper indicate the $ (in thousands) spent in advertising through the three medium

In [3]:
advertising = pd.read_csv('Advertising.csv', usecols=[1,2,3,4])
display(advertising)
advertising_X = advertising.loc[:,['TV','Radio','Newspaper']]
advertising_Y = advertising.loc[:,['Sales']]
display(advertising_X)
display(advertising_Y)

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,9.7
197,177.0,9.3,6.4,12.8
198,283.6,42.0,66.2,25.5


,TV,Radio,Newspaper
0,230.1,37.8,69.2
1,44.5,39.3,45.1
2,17.2,45.9,69.3
3,151.5,41.3,58.5
4,180.8,10.8,58.4
...,...,...,...
195,38.2,3.7,13.8
196,94.2,4.9,8.1
197,177.0,9.3,6.4
198,283.6,42.0,66.2


,Sales
0,22.1
1,10.4
2,9.3
3,18.5
4,12.9
...,...
195,7.6
196,9.7
197,12.8
198,25.5


## Train test split

In [7]:
(x_train,x_test,y_train,y_test)  = train_test_split(advertising_X,advertising_Y,test_size=0.20)
print(x_train.shape,
      y_train.shape,
      x_test.shape,
      y_test.shape)

(160, 3) (160, 1) (40, 3) (40, 1)


### Correlation matrix

In [9]:
df_train = pd.concat([x_train,y_train],axis=1)
df_train.corr()

,TV,Radio,Newspaper,Sales
TV,1.000000,0.034564,0.010971,0.769957
Radio,0.034564,1.000000,0.390945,0.569748
Newspaper,0.010971,0.390945,1.000000,0.213838
Sales,0.769957,0.569748,0.213838,1.000000


There is a modest correlation between Newspaper and Radio. 
Positive correlation exist between Sales and three features

### Scaling features
- recommended for gradient descent 

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)


### Use SGDregressor

In [13]:
from sklearn.linear_model import SGDRegressor



- `penalty = None` means no regularization
- `validation_fraction=0.1` means 10% of training set used for validation

In [30]:
model = SGDRegressor(loss='squared_error',penalty=None, tol = 0,max_iter=10000,validation_fraction=0.1,early_stopping=True,n_iter_no_change= 5)
model.fit(x_train_scaled,y_train)

C:\Users\soibamb\AppData\Local\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SGDRegressor(early_stopping=True, max_iter=10000, penalty=None, tol=0)

In [18]:
print('\n (intercept and slope)',model.intercept_,model.coef_)


 (intercept and slope) [13.92591012] [ 3.92335431  2.66556603 -0.01969904]


In [20]:
def model_summary(model,y_true,y_pred):
    
    print('R^{2} = ', r2_score(y_true,y_pred))
    MSE = mean_squared_error(y_true,y_pred)
    RSS = y_true.shape[0] * MSE
    RSE = np.sqrt(RSS/(y_true.shape[0]-2))
    print('RSS = ',RSS)
    print('RSE = ',RSE)
    print('MSE = ',MSE)

In [22]:
model_summary(model,y_test,y_pred = model.predict(x_test_scaled))

R^{2} =  0.9274167851015742
RSS =  86.02090838864586
RSE =  1.5045624332312304
MSE =  2.150522709716147


In [24]:
model_summary(model,y_train,y_pred = model.predict(x_train_scaled))

R^{2} =  0.886935576972711
RSS =  478.47743780708083
RSE =  1.7402121175215584
MSE =  2.9904839862942554


-Note that the metrics above are on the scaled data. 
-To interpret w.r.t to actual data, you have to inverse scale the data.